# Demo: principal component analysis on the wine data

Each section calls the problem's module in `src/` and shows its figures and key numbers; the
notebook has no logic of its own. `python -m src.run` produces the same numbers and figures as
files.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
from IPython.display import display

from src import p1, p2, p3, p4
from src.common.data import FEATURES
from src.common.fmt import fixed, sci


def show(module, res):
    for fig in module.figures(res).values():
        display(fig)
        plt.close(fig)

## P1: PCA from first principles

Standardise the 13 training features, build S with the divisor N - 1, and take its
eigendecomposition; scikit-learn's PCA is fitted only to compare against.

In [ ]:
r1 = p1.compute()
n1 = r1["numbers"]
print(r1["stdout"])
print()
print(f"|U^T U - I|_F          = {sci(n1['orthonormality'])}")
print(f"tr(S)                  = {fixed(n1['trace'])}")
print(f"|tr(S) - sum lambda_j| = {sci(n1['trace_minus_sum'])}")
diff = n1["library_diff"]
print(f"largest difference from scikit-learn, eigenvalues:  {sci(diff['eigenvalues'])}")
print(f"largest difference from scikit-learn, eigenvectors: {sci(diff['eigenvectors'])}")
print()
print(f"{'feature':<30}{'u_1':>9}{'u_2':>9}")
for name, u1, u2 in zip(FEATURES, *n1["loadings"].values(), strict=True):
    print(f"{name:<30}{fixed(u1):>9}{fixed(u2):>9}")
for pc, names in n1["top"].items():
    print(f"largest |loading| on {pc.upper()}: {', '.join(names)}")

## P2: variance captured vs. number of components

In [ ]:
r2 = p2.compute()
n2 = r2["numbers"]
show(p2, r2)
print(f"k_elbow = {n2['k_elbow']}, R(k_elbow) = {fixed(n2['R_elbow'])}")
print(", ".join(f"k_{tau} = {k}" for tau, k in n2["k_tau"].items()))
print(f"Kaiser criterion retains {n2['kaiser']} components")
print(f"max_k |J_train(k) - sum of discarded lambda_j| = {sci(n2['identity_residual'])}")
print()
print(f"{'k':>2}{'r_k':>9}{'R(k)':>9}{'J_train':>9}{'J_test':>9}")
rows = zip(n2["r"], n2["R"], n2["J_train"], n2["J_test"], strict=True)
for k, row in enumerate(rows, start=1):
    print(f"{k:>2}" + "".join(f"{fixed(v):>9}" for v in row))

## P3: effect of preprocessing, and the two-dimensional projection

In [ ]:
r3 = p3.compute()
n3 = r3["numbers"]
show(p3, r3)
print(f"{'variant':<18}{'r_1':>8}{'r_2':>8}{'r_1+r_2':>9}  {'top of PC1':<12}{'5-NN accuracy':>14}")
for name, v in n3["variants"].items():
    label = f"{name} ({p3.VARIANTS[name][0]})"
    print(
        f"{label:<18}{fixed(v['r1']):>8}{fixed(v['r2']):>8}{fixed(v['r1_plus_r2']):>9}  "
        f"{v['pc1_feature']:<12}{fixed(v['accuracy']):>14}"
    )
print()
print("largest raw sample variances (V1):")
for item in n3["raw_variance"]:
    print(f"  {item['feature']:<20}{fixed(item['variance']):>12}")

## P4: PCA versus feature selection for a downstream classifier

In [ ]:
r4 = p4.compute()
n4 = r4["numbers"]
show(p4, r4)
print(f"{'k':>2}{'(A) PCA':>10}{'(B) Filter':>12}{'(C) Wrapper':>13}")
for i, k in enumerate(p4.K_VALUES):
    print(
        f"{k:>2}"
        + "".join(
            f"{fixed(n4['accuracy'][m][i]):>{w}}"
            for m, w in zip(p4.METHODS, (10, 12, 13), strict=True)
        )
    )
base = n4["baseline"]
print(f"baseline, all 13 features: {fixed(base['accuracy'])} ({base['correct']} of {n4['n_test']})")
reached = ", ".join(f"{p4.METHODS[m]} k = {k}" for m, k in n4["within_one"].items())
print(f"smallest k within one test sample of the baseline: {reached}")
print()
print("F-score ranking, best first:")
for rank, name in enumerate(n4["f_ranking"], start=1):
    print(f"{rank:>3}  {name:<30}{fixed(n4['f_scores'][FEATURES.index(name)]):>10}")
print()
for k, sets in n4["chosen"].items():
    print(f"k = {k}")
    print("  (B) filter: ", ", ".join(sets["filter"]))
    print("  (C) wrapper:", ", ".join(sets["wrapper"]))
    print(f"  shared ({len(sets['shared'])}):  ", ", ".join(sets["shared"]))